In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

In [20]:
import torchvision
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor
from torchvision import models

In [3]:
from torch.utils.data import DataLoader

In [4]:
device="cuda" if torch.cuda.is_available() else  "cpu"
device

'cuda'

In [5]:
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = correct / len(y_pred) * 100
    return acc

In [80]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

# 2. TEST TRANSFORMS (Clean Exam + Normalize)
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # Added back! Test data must be normalized the exact same way as train data
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [81]:
train_dir=r"C:\Important\hotdog-nothotdog\hotdog-nothotdog\train"
test_dir=r"C:\Important\hotdog-nothotdog\hotdog-nothotdog\test"

In [78]:
print("Loading images from disk...")
train_data = datasets.ImageFolder(root=train_dir, transform=train_transforms)
test_data = datasets.ImageFolder(root=test_dir, transform=test_transforms)

Loading images from disk...


In [82]:
print(f"\n--- Dataset Info ---")
print(f"Classes found: {train_data.classes}")
print(f"Total Training Images: {len(train_data)}")
print(f"Total Testing Images: {len(test_data)}")


--- Dataset Info ---
Classes found: ['hotdog', 'nothotdog']
Total Training Images: 4242
Total Testing Images: 400


In [ ]:
class Hotdog(nn.Module):
    def __init__(self, input_shape:int,hidden_layers:int,output_shape:int):
        super().__init__()
        self.block1=nn.Sequential(
            nn.Conv2d(in_channels=input_shape,out_channels=16,kernel_size=3,stride=1,padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(in_channels=16,out_channels=16,kernel_size=3,stride=1,padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            #nn.Dropout2d(0.1),
            nn.MaxPool2d(kernel_size=2,stride=2)            
        )
        self.block2=nn.Sequential(
            nn.Conv2d(in_channels=16,out_channels=32,kernel_size=3,stride=1,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(in_channels=32,out_channels=32,kernel_size=3,stride=1,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            #nn.Dropout2d(0.1),
            nn.MaxPool2d(2,2) 
        )
        self.block3=nn.Sequential(
            nn.Conv2d(in_channels=32,out_channels=32,kernel_size=3,stride=1,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(in_channels=32,out_channels=32,kernel_size=3,stride=1,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            #nn.Dropout2d(0.1),
            nn.MaxPool2d(2,2) 
        )
        self.classifier=nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.7),
            nn.Linear(in_features=25088, out_features=output_shape)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
       # print(x.shape) 
        x = self.classifier(x)
        return x
    

torch.manual_seed(42)

model_0=Hotdog(input_shape=3,hidden_layers=32,output_shape=1).to(device)
model_0
##this given 70 percent accuracy on test data, which is pretty good for a first try. Let's see if we can do better with a pretrained model.

Hotdog(
  (block1): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(32, 32, k

In [85]:
model_0 = models.resnet18(pretrained=True)
for param in model_0.parameters():
    param.requires_grad = False
model_0.fc = nn.Linear(model_0.fc.in_features, 1)
model_0 = model_0.to(device)
model_0

c:\Users\yashy\anaconda3\envs\pytorch_env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\yashy\anaconda3\envs\pytorch_env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [84]:
loss_fn=nn.BCEWithLogitsLoss()

optimizer=torch.optim.Adam(model_0.parameters(),lr=0.001, weight_decay=1e-4)

from torch.optim.lr_scheduler import StepLR
scheduler = StepLR(optimizer, step_size=10, gamma=0.5)

In [64]:
BATCH_SIZE=32


train_dataloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

In [65]:
def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               accuracy_fn,
               device: torch.device = device):
    train_loss, train_acc = 0, 0
    model.to(device)
    for batch, (X, y) in enumerate(data_loader):
        # Send data to GPU
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        y_pred = model(X).squeeze()

        # 2. Calculate loss
        loss = loss_fn(y_pred, y.float())
        train_loss += loss.item()
        train_acc += accuracy_fn(y_true=y,
                                 y_pred=torch.round(torch.sigmoid(y_pred))) # Go from logits -> pred labels


        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

    # Calculate loss and accuracy per epoch and print out what's happening
    train_loss /= len(data_loader)
    train_acc /= len(data_loader)
    print(f"Train loss: {train_loss:.5f} | Train accuracy: {train_acc:.2f}%")

def test_step(data_loader: torch.utils.data.DataLoader,
              model: torch.nn.Module,
              loss_fn: torch.nn.Module,
              accuracy_fn,
              device: torch.device = device):
    test_loss, test_acc = 0, 0
    model.to(device)
    model.eval() # put model in eval mode
    # Turn on inference context manager
    with torch.inference_mode(): 
        for X, y in data_loader:
            # Send data to GPU
            X, y = X.to(device), y.to(device)
            
            # 1. Forward pass
            test_pred = model(X).squeeze()
            
            # 2. Calculate loss and accuracy
            test_loss += loss_fn(test_pred, y.float())
            test_acc += accuracy_fn(y_true=y,
                y_pred=torch.round(torch.sigmoid(test_pred)) # Go from logits -> pred labels
            )
        
        # Adjust metrics and print out
        test_loss /= len(data_loader)
        test_acc /= len(data_loader)
        print(f"Test loss: {test_loss:.5f} | Test accuracy: {test_acc:.2f}%\n")

In [86]:
epochs=30

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs} ----------------------")
    
    train_step(
        model=model_0,
        data_loader=train_dataloader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        accuracy_fn=accuracy_fn
    )

    test_step(
        model=model_0,
        data_loader=test_dataloader,
        loss_fn=loss_fn,
        accuracy_fn=accuracy_fn
    )

    scheduler.step()
    print(f"Current LR: {scheduler.get_last_lr()[0]}")


Epoch 1/30 ----------------------
Train loss: 0.74955 | Train accuracy: 48.09%
Test loss: 0.72810 | Test accuracy: 54.33%

Current LR: 0.001

Epoch 2/30 ----------------------
Train loss: 0.74549 | Train accuracy: 48.77%
Test loss: 0.72810 | Test accuracy: 54.33%

Current LR: 0.001

Epoch 3/30 ----------------------
Train loss: 0.74575 | Train accuracy: 48.82%
Test loss: 0.72810 | Test accuracy: 54.33%

Current LR: 0.001

Epoch 4/30 ----------------------
Train loss: 0.75180 | Train accuracy: 48.10%
Test loss: 0.72810 | Test accuracy: 54.33%

Current LR: 0.001

Epoch 5/30 ----------------------
Train loss: 0.74830 | Train accuracy: 48.70%
Test loss: 0.72810 | Test accuracy: 54.33%

Current LR: 0.001

Epoch 6/30 ----------------------
Train loss: 0.74757 | Train accuracy: 48.74%
Test loss: 0.72810 | Test accuracy: 54.33%

Current LR: 0.001

Epoch 7/30 ----------------------
Train loss: 0.74801 | Train accuracy: 48.21%
Test loss: 0.72810 | Test accuracy: 54.33%

Current LR: 0.001

Epoch

KeyboardInterrupt: 